# Lab 10 – Regression with Multi-Input Variables
### CS201L: Artificial Intelligence Laboratory — IIT Dharwad
**Dataset:** AirQuality.csv  
**Task:** Predict CO concentration (mg/m³) using Ridge Regression and Neural Networks

## 1. Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import mean_squared_error
import torch
import torch.nn as nn
import torch.optim as optim
import warnings
warnings.filterwarnings('ignore')

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

## 2. Load & Preprocess Dataset

In [ ]:
# Load dataset — update path if needed
df = pd.read_csv('AirQuality.csv', sep=';', decimal=',')

# Drop non-feature columns
df.drop(columns=['Date', 'Time'], inplace=True, errors='ignore')

# Replace -200 sentinel values (missing) with NaN, then drop
df.replace(-200, np.nan, inplace=True)
df.dropna(inplace=True)

# Rename target column for convenience
# The target is 'CO(GT)' in AirQuality.csv
df.rename(columns={'CO(GT)': 'CO'}, inplace=True)

print(f"Dataset shape after cleaning: {df.shape}")
print("\nColumn names:")
print(df.columns.tolist())
df.head()

## 3. Data Splitting (60% Train / 20% Val / 20% Test)

In [ ]:
from sklearn.model_selection import train_test_split

# First split: 60% train, 40% temp
train_df, temp_df = train_test_split(df, test_size=0.40, random_state=42)

# Second split: 50% of temp → val (20% overall), 50% → test (20% overall)
val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=42)

print(f"Train size : {len(train_df):>5}  ({len(train_df)/len(df)*100:.1f}%)")
print(f"Val   size : {len(val_df):>5}  ({len(val_df)/len(df)*100:.1f}%)")
print(f"Test  size : {len(test_df):>5}  ({len(test_df)/len(df)*100:.1f}%)")

## 4. Correlation Analysis & Feature Selection

In [ ]:
# Compute correlation of each input feature with target 'CO'
correlations = train_df.corr()['CO'].drop('CO')

print("Correlation with CO concentration:")
print(correlations.sort_values(key=abs, ascending=False).to_string())

# Best single feature
x_best = correlations.abs().idxmax()
print(f"\n>>> Feature with highest absolute correlation: '{x_best}'  "
      f"(r = {correlations[x_best]:.4f})")

# Top-2 features
top2_features = correlations.abs().nlargest(2).index.tolist()
print(f">>> Top-2 features: {top2_features}")

In [ ]:
# Bar chart of correlations
fig, ax = plt.subplots(figsize=(10, 4))
sorted_corr = correlations.sort_values(key=abs, ascending=True)
colors = ['steelblue' if f not in top2_features else 'tomato' for f in sorted_corr.index]
sorted_corr.plot(kind='barh', ax=ax, color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Feature Correlation with CO Concentration')
ax.set_xlabel('Pearson Correlation Coefficient')
plt.tight_layout()
plt.show()

In [ ]:
# Prepare single-feature splits (for Ridge)
X_train = train_df[[x_best]]
y_train = train_df['CO']

X_val = val_df[[x_best]]
y_val = val_df['CO']

X_test = test_df[[x_best]]
y_test = test_df['CO']

# Prepare top-2 feature splits (for Neural Network)
X_train_top2 = train_df[top2_features]
X_val_top2   = val_df[top2_features]
X_test_top2  = test_df[top2_features]

## 5. Ridge Regression (Polynomial Degree = 4)

In [ ]:
# Polynomial transformation (degree = 4)
poly = PolynomialFeatures(degree=4)
X_train_poly = poly.fit_transform(X_train)
X_val_poly   = poly.transform(X_val)
X_test_poly  = poly.transform(X_test)

# Ridge Regression (alpha = 1)
ridge = Ridge(alpha=1)
ridge.fit(X_train_poly, y_train)

# Predictions
y_train_pred_ridge = ridge.predict(X_train_poly)
y_val_pred_ridge   = ridge.predict(X_val_poly)
y_test_pred_ridge  = ridge.predict(X_test_poly)

# RMSE
rmse_train_ridge = np.sqrt(mean_squared_error(y_train, y_train_pred_ridge))
rmse_val_ridge   = np.sqrt(mean_squared_error(y_val,   y_val_pred_ridge))
rmse_test_ridge  = np.sqrt(mean_squared_error(y_test,  y_test_pred_ridge))

print("=== Ridge Regression (Degree=4, alpha=1) ===")
print(f"Train RMSE : {rmse_train_ridge:.4f}")
print(f"Val   RMSE : {rmse_val_ridge:.4f}")
print(f"Test  RMSE : {rmse_test_ridge:.4f}")

In [ ]:
# Plot Ridge fit curve
x_plot = np.linspace(X_train[x_best].min(), X_train[x_best].max(), 300).reshape(-1, 1)
x_plot_poly = poly.transform(x_plot)
y_plot = ridge.predict(x_plot_poly)

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(X_train[x_best], y_train, alpha=0.3, s=10, label='Train data', color='steelblue')
ax.scatter(X_val[x_best],   y_val,   alpha=0.3, s=10, label='Val data',   color='orange')
ax.plot(x_plot, y_plot, color='red', linewidth=2, label='Ridge fit (degree=4)')
ax.set_xlabel(x_best)
ax.set_ylabel('CO Concentration (mg/m³)')
ax.set_title(f'Ridge Regression – {x_best} vs CO')
ax.legend()
plt.tight_layout()
plt.show()

## 6. Neural Network Regression (PyTorch) – Top-2 Features

In [ ]:
# --- Normalize features (important for SGD convergence) ---
from sklearn.preprocessing import StandardScaler

scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_train_scaled = scaler_X.fit_transform(X_train_top2)
X_val_scaled   = scaler_X.transform(X_val_top2)
X_test_scaled  = scaler_X.transform(X_test_top2)

y_train_scaled = scaler_y.fit_transform(y_train.values.reshape(-1, 1))
y_val_scaled   = scaler_y.transform(y_val.values.reshape(-1, 1))
y_test_scaled  = scaler_y.transform(y_test.values.reshape(-1, 1))

# Convert to PyTorch tensors
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train_scaled, dtype=torch.float32)

X_val_tensor = torch.tensor(X_val_scaled, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val_scaled, dtype=torch.float32)

X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test_scaled, dtype=torch.float32)

print("Tensors ready.")
print(f"  X_train: {X_train_tensor.shape}, y_train: {y_train_tensor.shape}")

In [ ]:
# --- Neural Network Definition ---
class Net(nn.Module):
    def __init__(self, hidden_size):
        super(Net, self).__init__()
        self.fc1  = nn.Linear(2, hidden_size)
        self.tanh = nn.Tanh()
        self.fc2  = nn.Linear(hidden_size, 1)

    def forward(self, x):
        x = self.tanh(self.fc1(x))
        x = self.fc2(x)
        return x

In [ ]:
# --- Training Loop ---
hidden_sizes = [16, 32, 64]
EPOCHS = 500
LR = 0.001

results = {}   # stores {hidden_size: {'train_losses', 'val_losses', 'model'}}

for h in hidden_sizes:
    torch.manual_seed(42)
    model     = Net(h)
    criterion = nn.MSELoss()
    optimizer = optim.SGD(model.parameters(), lr=LR)

    train_losses = []
    val_losses   = []

    for epoch in range(EPOCHS):
        # --- Train ---
        model.train()
        optimizer.zero_grad()
        outputs = model(X_train_tensor)
        loss    = criterion(outputs, y_train_tensor)
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())

        # --- Validate ---
        model.eval()
        with torch.no_grad():
            val_out  = model(X_val_tensor)
            val_loss = criterion(val_out, y_val_tensor)
        val_losses.append(val_loss.item())

    results[h] = {
        'train_losses': train_losses,
        'val_losses':   val_losses,
        'model':        model
    }
    print(f"Hidden={h:>2}  |  Final Train Loss: {train_losses[-1]:.6f}  "
          f"Val Loss: {val_losses[-1]:.6f}")

In [ ]:
# --- Plot Training & Validation Loss vs Epochs ---
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
colors = {'train': 'steelblue', 'val': 'tomato'}

for ax, h in zip(axes, hidden_sizes):
    ax.plot(results[h]['train_losses'], color=colors['train'], label='Train Loss')
    ax.plot(results[h]['val_losses'],   color=colors['val'],   label='Val Loss',   linestyle='--')
    ax.set_title(f'Hidden Size = {h}')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('MSE Loss')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('Training & Validation Loss vs Epochs', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# --- RMSE Summary for all NN configurations ---
def compute_rmse(model, X_tensor, y_true_scaled, scaler_y):
    model.eval()
    with torch.no_grad():
        y_pred_scaled = model(X_tensor).numpy()
    y_pred = scaler_y.inverse_transform(y_pred_scaled)
    y_true = scaler_y.inverse_transform(y_true_scaled.numpy())
    return np.sqrt(mean_squared_error(y_true, y_pred))

print("=== Neural Network RMSE Summary ===")
print(f"{'Hidden':>6} | {'Train RMSE':>10} | {'Val RMSE':>10}")
print("-" * 34)

best_h, best_val_rmse = None, np.inf

for h in hidden_sizes:
    m = results[h]['model']
    rmse_tr = compute_rmse(m, X_train_tensor, y_train_tensor, scaler_y)
    rmse_vl = compute_rmse(m, X_val_tensor,   y_val_tensor,   scaler_y)
    print(f"{h:>6} | {rmse_tr:>10.4f} | {rmse_vl:>10.4f}")
    if rmse_vl < best_val_rmse:
        best_val_rmse = rmse_vl
        best_h = h

print(f"\n>>> Best model: Hidden Size = {best_h}  (Val RMSE = {best_val_rmse:.4f})")

In [ ]:
# --- Evaluate best model on Test Set ---
best_model = results[best_h]['model']
rmse_test_nn = compute_rmse(best_model, X_test_tensor, y_test_tensor, scaler_y)

print(f"=== Best Neural Network (Hidden={best_h}) – Test RMSE ===")
print(f"Test RMSE : {rmse_test_nn:.4f}")

## 7. 3D Best-Fit Surface (Linear Regression with Top-2 Features)

In [ ]:
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

# Train Linear Regression on top-2 features
lin_model = LinearRegression()
lin_model.fit(X_train_top2, y_train)

f1, f2 = X_train_top2.columns

# Create grid over feature space
x1 = np.linspace(X_train_top2[f1].min(), X_train_top2[f1].max(), 100)
x2 = np.linspace(X_train_top2[f2].min(), X_train_top2[f2].max(), 100)
X1_grid, X2_grid = np.meshgrid(x1, x2)

# Predict surface
grid_points = np.c_[X1_grid.ravel(), X2_grid.ravel()]
Z_pred = lin_model.predict(grid_points).reshape(X1_grid.shape)

# 3D Plot
fig = plt.figure(figsize=(10, 7))
ax  = fig.add_subplot(111, projection='3d')

surface = ax.plot_surface(X1_grid, X2_grid, Z_pred, cmap='viridis', alpha=0.7)
ax.scatter(X_train_top2[f1], X_train_top2[f2], y_train,
           color='red', s=10, alpha=0.4, label='Training Data')

ax.set_xlabel(f1)
ax.set_ylabel(f2)
ax.set_zlabel('CO Concentration (mg/m³)')
ax.set_title('3D Best-Fit Surface (Linear Regression)')
fig.colorbar(surface, ax=ax, shrink=0.5, aspect=10)
ax.view_init(elev=30, azim=45)
ax.legend()
plt.tight_layout()
plt.show()

w1, w2 = lin_model.coef_
b      = lin_model.intercept_
print(f"Regression plane: CO = {w1:.4f}*{f1} + {w2:.4f}*{f2} + {b:.4f}")

## 8. Final Summary

In [ ]:
print("========================================")
print("          FINAL RESULTS SUMMARY         ")
print("========================================")
print(f"Best single feature (x_best)  : {x_best}")
print(f"Top-2 features                : {top2_features}")
print()
print("--- Ridge Regression (Degree=4, α=1) ---")
print(f"  Train RMSE : {rmse_train_ridge:.4f}")
print(f"  Val   RMSE : {rmse_val_ridge:.4f}")
print(f"  Test  RMSE : {rmse_test_ridge:.4f}")
print()
print(f"--- Neural Network (Best: Hidden={best_h}) ---")
rmse_train_best = compute_rmse(best_model, X_train_tensor, y_train_tensor, scaler_y)
print(f"  Train RMSE : {rmse_train_best:.4f}")
print(f"  Val   RMSE : {best_val_rmse:.4f}")
print(f"  Test  RMSE : {rmse_test_nn:.4f}")
print("========================================")